## Задача

# Състезание за SAT-класификатор: Прогнозиране на удовлетворимост на Булеви формули в 3-КНФ

**Въведение**: Това състезание предизвиква участниците да прогнозират удовлетворимостта на Булеви формули в 3-КНФ (3-та конюнктивна нормална форма), използвайки традиционни подходи от Oбработката на естествен език (NLP) и инженеринг на признаци.

## I. Общ преглед на проблема

Предоставен е JSON файл, съдържащ синтетично генерирани Булеви формули в 3-КНФ, съхранени в тренировъчния набор "train.json". Всяка формула представлява проблем за Булева удовлетворимост със специфични ограничения. Променливите в набора от данни са:

* **formula**: Символен низ, представляващ Булева формула в 3-КНФ, използваща логически символи (∧ за И, ∨ за ИЛИ, ¬ за НЕ). Всяка формула съдържа между 5 и 20 променливи (x1, x2, ..., x20) и между 40 и 100 клаузи, с точно 3 литерала във всяка клауза.
* **label**: Истинската стойност за удовлетворимост. Стойност 1 показва, че формулата е удовлетворима (съществува присвояване на стойности на променливите, което удовлетворява всички клаузи); стойност 0 показва, че е неудовлетворима.

**Примерни формули**:
- `"(x1 ∨ ¬x3 ∨ x7) ∧ (¬x2 ∨ x4 ∨ x5) ∧ (x1 ∨ x2 ∨ x3)"`
- `"(¬x1 ∨ ¬x2 ∨ ¬x3) ∧ (x1 ∨ x2 ∨ x3) ∧ (¬x1 ∨ x2 ∨ x3) ∧ (x1 ∨ ¬x2 ∨ x3)"`

Тренировъчният набор съдържа 9000 етикетирани формули, а валидационният набор съдържа 1000 етикетирани формули. Тестовият набор има приблизително 2000 формули с подобно разпределение.

## II. Набор от данни

**Тренировъчно множество**: train.json (9,000 примера)
**Валидационно множество**: val.json (1,000 примера)

## III. Изисквания към задачата

Създайте модел за двоична класификация, който да прогнозира удовлетворимостта на Булеви формули в 3-КНФ, като използвате само **Логистична регресия** с традиционни подходи от Обработка на естествен език (NLP) и ръчен инженеринг на признаци. Специфични ограничения:

**Позволени методи за извличане на характеристики:**
1. **Текстови характеристики**: TF-IDF векторизация, Торба с думи (Bag-of-Words), n-грамни признаци (униграми, биграми, триграми)
2. **Ръчно извлечени структурни характеристики** от синтактичния анализ на формулата:
  - Брой клаузи, брой уникални променливи.
  - Статистики за дължината на клаузите, честота на срещане на литералите.
  - Модели на отрицания, статистики за съвместно срещане на променливи.
  - Мерки за сходство между клаузите, метрики за разпределение на променливите.

**Изискване за модел за машинно обучение:**
- **Само Логистична регресия** (от sklearn.linear_model.LogisticRegression)
- Позволена е настройка на хиперпараметри (C, penalty).

**Изисквания за инженеринга на характеристики:**
1. Имплементирайте поне 2 различни подхода за извличане на признаци.
2. Комбинирайте текстови и структурни признаци.
3. Приложете подходяща предварителна обработка (нормализация, подбор/селекция на характеристики).
4. Документирайте логиката зад инженеринга на признаци в коментари в кода.

## IV. Формат на предаване

Изпратете компресиран файл с име **submission.zip**, съдържащ:

1. **submission_model.py**: Пълна имплементация на модела, включваща:
  - Функция `train_model()`, която обучава модела върху train.json и val.json.
  - Функция `predict(formula_list)`, която връща прогнози за списък от формули (представени като символни низове).
  - Всички конвейери (pipelines) за извличане на характеристики и модела за Логистична регресия.

2. **submission_model.pkl**: Сериализиран (запазен) обучен модел с помощта на joblib.

## V. Метрики за оценка

**Основна метрика**: Резултат ROC AUC, както е имплементирана в sklearn.metrics.roc_auc_score.


## Създаване на набор от данни

In [ ]:
pip install python-sat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 21.1 MB/s eta 0:00:00


In [ ]:
import random
import json
import argparse
from pysat.solvers import Glucose3

random.seed(42)

def generate_3cnf_formula(num_vars, num_clauses):
    formula = []
    for _ in range(num_clauses):
        clause = []
        vars_in_clause = random.sample(range(1, num_vars + 1), 3)
        for var in vars_in_clause:
            literal = f"x{var}" if random.random() < 0.5 else f"~x{var}"
            clause.append(literal)
        formula.append(f"({' v '.join(clause)})")
    return " ^ ".join(formula)


def formula_to_cnf_list(formula_str):
    clauses = []
    for part in formula_str.split("^"):
        part = part.strip()[1:-1]  # remove parentheses
        literals = part.split("v")
        clause = []
        for lit in literals:
            lit = lit.strip()
            if lit.startswith("~"):
                clause.append(-int(lit[2:]))  # ¬x5 -> -5
            else:
                clause.append(int(lit[1:]))  # x5 -> 5
        clauses.append(clause)
    return clauses


def check_satisfiability(clauses):
    solver = Glucose3()
    for clause in clauses:
        solver.add_clause(clause)
    is_sat = solver.solve()
    solver.delete()
    return int(is_sat)


def create_dataset(num_formulas=10_000):
    dataset = []
    for _ in range(num_formulas):
        num_vars = random.randint(5, 20)
        num_clauses = random.randint(40, 100)
        formula_str = generate_3cnf_formula(num_vars, num_clauses)
        clauses = formula_to_cnf_list(formula_str)
        label = check_satisfiability(clauses)
        dataset.append({"formula": formula_str, "label": label})
    random.shuffle(dataset)
    train = dataset[:int(num_formulas*0.9)]
    val = dataset[int(num_formulas*0.9):]

    with open("train.json", "w") as f:
        json.dump(train, f, indent=2)
    with open("val.json", "w") as f:
        json.dump(val, f, indent=2)

In [ ]:
create_dataset()

In [ ]:
ls

sample_data/  train.json  val.json


In [ ]:
!head train.json

[
  {
    "formula": "(x6 v ~x3 v ~x15) ^ (x6 v ~x10 v x16) ^ (~x16 v x15 v ~x6) ^ (x5 v ~x3 v ~x8) ^ (~x14 v x1 v x7) ^ (~x7 v x9 v x6) ^ (x11 v ~x15 v x5) ^ (x1 v x2 v ~x12) ^ (x1 v ~x13 v ~x7) ^ (x5 v ~x1 v ~x8) ^ (~x13 v x4 v x10) ^ (~x1 v ~x17 v x12) ^ (x10 v ~x7 v x13) ^ (x5 v x7 v x1) ^ (x9 v ~x8 v ~x11) ^ (x9 v ~x12 v x15) ^ (~x8 v x17 v ~x4) ^ (~x8 v ~x7 v ~x3) ^ (x4 v ~x7 v x13) ^ (x10 v x15 v x6) ^ (x17 v x4 v ~x14) ^ (x2 v x4 v ~x15) ^ (~x6 v x5 v ~x12) ^ (x4 v ~x7 v x9) ^ (~x8 v ~x5 v ~x11) ^ (x7 v ~x6 v ~x1) ^ (~x1 v x6 v ~x15) ^ (x1 v ~x10 v x7) ^ (x8 v x3 v ~x1) ^ (~x3 v x16 v x10) ^ (~x16 v ~x14 v ~x17) ^ (x4 v x16 v x13) ^ (~x5 v x6 v ~x17) ^ (~x4 v ~x9 v x1) ^ (x2 v x11 v ~x16) ^ (~x11 v x14 v x8) ^ (x7 v ~x2 v x8) ^ (x15 v ~x6 v x16) ^ (x4 v x11 v ~x12) ^ (x17 v ~x12 v x16) ^ (x11 v ~x2 v x15) ^ (~x13 v ~x1 v x3) ^ (~x8 v ~x11 v ~x4) ^ (~x6 v x10 v x2) ^ (~x10 v ~x9 v x17) ^ (x15 v x14 v x10) ^ (x2 v x16 v x15) ^ (x14 v x15 v x9) ^ (~x16 v ~x5 v ~x10) ^ (~x5 v ~x4 v

## Baseline model

In [ ]:
import json
import argparse
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

def load_data(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    return pd.DataFrame(data)


def preprocess(text):
    # Optional: Remove parentheses, normalize spacing
    return text.replace("(", "").replace(")", "")


def train():
    train = load_data("train.json")
    val = load_data("val.json")

    train["formula"] = train["formula"].apply(preprocess)
    val["formula"] = val["formula"].apply(preprocess)

    # =============================================================================
    # COMPETITION ZONE: MODIFY CODE BELOW THIS LINE
    # =============================================================================
    # You can:
    # - Create new features or feature extractors
    # - Tune classifier hyperparameters
    # - Add different classifiers (RandomForest, SVM, etc.)
    # - Experiment with different vectorizers or tokenization strategies
    # - Combine multiple models or use ensemble methods
    # - Add preprocessing steps
    #
    # Goal: Improve the F1 score and ROC AUC for 3-SAT satisfiability prediction
    # =============================================================================

    # TF-IDF using custom token splitting on AND and OR
    vectorizer = TfidfVectorizer(
        token_pattern=r"~?x\d+",  # Extract literals
    )
    X = vectorizer.fit_transform(train["formula"])
    y = train["label"]

    clf = LogisticRegression()
    clf.fit(X, y)

    X_val = vectorizer.transform(val["formula"])
    y_val = val["label"]


    # =============================================================================
    # НЕ ПРОМЕНЯЙ КОДА ПОД ТОЗИ РЕД!
    # =============================================================================

    y_pred = clf.predict(X_val)
    pred_proba = clf.predict_proba(X_val)[:, 1]

    roc_auc = roc_auc_score(val["label"], pred_proba)

    print("Classification Report:")
    print(classification_report(y_val, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_val, y_pred))

    print(f"ROC AUC Score: {roc_auc:.4f}")

train()

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.82      0.81       613
           1       0.71      0.68      0.69       387

    accuracy                           0.77      1000
   macro avg       0.75      0.75      0.75      1000
weighted avg       0.77      0.77      0.77      1000

Confusion Matrix:
[[504 109]
 [124 263]]
ROC AUC Score: 0.8397


In [ ]:
train = load_data("train.json")
val = load_data("val.json")

In [ ]:
train.label.value_counts()

,count
label,
0,5501
1,3499


In [ ]:
val.label.value_counts()

,count
label,
0,613
1,387
